# Lingolate lexicon build (schema v3)

Runs on a **standard CPU runtime** (no GPU needed). Downloads the open sources, builds the master lexicon, exports one pack per language pair, uploads them to a GitHub Release and commits `manifest-v3.json`.

Needs the Colab secret **`GH_TOKEN`** (repo `contents: write`). The runtime disconnects itself at the end to save compute units.

In [ ]:
VERSION = '2026.11.1'
TAG = f'packs-v{VERSION}'
import os, time
T0 = time.time()
!git clone -q --depth 1 https://github.com/hudaiapa88/lingolate-dict-data.git /content/ldd
%cd /content/ldd
!pip install -q -r scripts/requirements.txt
os.environ['PYTHONIOENCODING'] = 'utf-8'
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# Sources: Open Multilingual Wordnet, TUFS, PanLex (all 27 langs), Tatoeba
!cd scripts && python download_omw.py > /content/dl_omw.log 2>&1; tail -2 /content/dl_omw.log
!cd scripts && python download_tufs.py > /content/dl_tufs.log 2>&1; tail -2 /content/dl_tufs.log
!cd scripts && python download_panlex.py --langs ar,bg,cs,da,de,el,en,es,fi,fr,hi,id,it,ja,ko,lt,nl,no,pl,pt,ru,sv,th,tr,uk,vi,zh > /content/dl_panlex.log 2>&1; tail -3 /content/dl_panlex.log
!mkdir -p sources_cache/tatoeba && cd sources_cache/tatoeba && for f in sentences_detailed links tags user_languages sentences_CC0; do curl -sSL --retry 5 -o $f.tar.bz2 https://downloads.tatoeba.org/exports/$f.tar.bz2 && bzip2 -t $f.tar.bz2 && echo "$f ok"; done
!du -sh sources_cache/*

In [ ]:
!python scripts/build_tatoeba_examples.py > /content/tatoeba.log 2>&1; tail -2 /content/tatoeba.log
!python scripts/build_lexicon.py 2>&1 | tail -12

In [ ]:
!python scripts/export_packs.py --version {VERSION} > /content/export.log 2>&1; grep -E 'en-tr|de-tr|ko-tr|exported' /content/export.log
!python scripts/generate_pack_manifest.py --version {VERSION} --tag {TAG}

In [ ]:
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
!python scripts/upload_packs.py --tag {TAG} --version {VERSION} 2>&1 | tail -4
print(f'total {(time.time() - T0) / 60:.0f} min')

In [ ]:
# Stop the runtime so it does not keep using compute units.
from google.colab import runtime
runtime.unassign()